# Entity ***Swaps***
+ Layer bronze
+ Priority 2
---
### Imports

In [0]:
%run ../00_functions/medallion_functions

In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
dbutils.widgets.text("layer", "bronze")
dbutils.widgets.text("entity_name", "swaps")
dbutils.widgets.text("load_pattern", "1")
dbutils.widgets.text("SUBGRAPH_URL", "https://gateway.thegraph.com/api/a5bbc98aac75dac555f9aba8747c7e2e/subgraphs/id/5zvR82QoaXYFyDEKLZ9t6v9adgnptxYpKpSbxtgVENFV")

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")

### Variables

In [0]:
PAGE_SIZE = 1000
swaps_list = []
query_variables= {
    "pool_id": "",
    "skip": 0
}

In [0]:
swaps_query = """
query SwapsQuery($pool_id: String!, $skip: Int!){
  swaps(
    where: {
      pool: $pool_id
    }
    first: 1000,
    skip: $skip, 
    orderBy: timestamp, 
    orderDirection: desc, 
  ) {
    timestamp
    transaction {
      id
    }
    pool {
      id
    }
    token0
    token1
    amount0
    amount1
    amountUSD
    sender
    recipient
    tick
  }
}
"""

### Execution

In [0]:
pools_df = spark.read.table("uniswap.bronze.pools").filter("liquidity != '0' and cast(txCount as int) >1000 ").limit(100)

In [0]:
display(pools_df)

In [0]:
for row in pools_df.collect():
    print(f"*INFO*: Processing swaps from pool: {row.token0["symbol"]}/{row.token1["symbol"]} | {row.id}.")
    query_variables.update({"pool_id": row.id})

    while True:
        print(f"*INFO*: Skip rows amount: {query_variables["skip"]}.")
        response = requests.post(SUBGRAPH_URL, json={"query": swaps_query, "variables": query_variables})
        swaps_list.extend(response.json()["data"][entity_name])

        time.sleep(0.3)

        rows_loaded = len(response.json()["data"][entity_name])
        print(f"*INFO*: Number of rows loaded: {rows_loaded}.")

        if rows_loaded == PAGE_SIZE:
            query_variables.update({"skip": query_variables["skip"] + PAGE_SIZE}) 
        elif rows_loaded == 0: 
            break
        else:
            query_variables.update({"skip": query_variables["skip"] + rows_loaded})  

    query_variables.update({"skip": 0}) 

In [0]:
swaps_df = spark.createDataFrame(swaps_list)
swaps_df = swaps_df.withColumn("load_timestamp", current_timestamp())

In [0]:
display(swaps_df)

### Save & exit

In [0]:
load_result = load_entity(swaps_df,
                        entity_name,
                        load_pattern,
                        layer
                        #,
                        )

In [0]:
dbutils.notebook.exit(load_result)